In [17]:
"""
의학 텍스트 분류기와 프롬프트를 통합한 파일 (GPT API 형식)
"""
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from prompts.medical_prompts import CCPrompts
from prompts.medical_prompts import TreatmentPrompts
from prompts.medical_prompts import TherapyPrompts
# from prompts.medical_prompts import PresentIllnessPrompts

from prompts.PI_prompts import PresentIllnessPrompts_ver2
from prompts.medical_prompts import NumericPrompts

from rate_limiter.rate_limiter import RateLimiter

from tqdm.asyncio import tqdm as tqdm_asyncio

In [18]:
pd.set_option('display.max_columns', None)

In [28]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_85013/962184981.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


In [29]:
df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI']]

In [30]:
df = df.sample(100)

In [39]:
class Config:
    # 실제 OpenAI API 키로 교체하거나 환경변수로부터 로드하세요.
    API_KEY = api_key
    # MODEL_NAME = "gpt-4o"
    MODEL_NAME = "o3-mini"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 100
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 4
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"
    
    # o1 모델 API 제한 반영 (약간의 여유를 둠)
    RPM_LIMIT = 4800  # 5,000 RPM
    TPM_LIMIT = 3800000  # 4,000,000 TPM
    TPD_LIMIT = 38000000  # 40,000,000 TPD
    INDEX_COLUMNS = ['환자번호', '날짜']  # 명시적 인덱스 컬럼 정의

#############################################
# 로깅 설정 함수
#############################################

def setup_logging(log_file=Config.LOG_FILE):
    """로깅 설정을 초기화하는 함수"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# 초기 로거 생성 (설정은 아직 적용되지 않음)
logger = logging.getLogger(__name__)


#############################################
# 체크포인트 관리 클래스
#############################################

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
    
    def get_checkpoint_path(self, column: str) -> str:
        safe_column = column.replace("/", "_").replace("\\", "_")
        return os.path.join(self.checkpoint_dir, f"{safe_column}_checkpoint.parquet")

    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            df_to_save = df.copy()
            # 만약 인덱스가 MultiIndex인 경우, 인덱스를 리셋 후 각 컬럼에서 Timestamp 객체를 문자열로 변환
            if isinstance(df_to_save.index, pd.MultiIndex):
                df_to_save = df_to_save.reset_index()
                for col in df_to_save.columns:
                    # dtype이 datetime64[ns] 뿐만 아니라 object 타입에 Timestamp가 있을 수 있으므로 적용
                    df_to_save[col] = df_to_save[col].apply(
                        lambda x: str(x) if isinstance(x, pd.Timestamp) else x
                    )
                # 인덱스 재설정 시 Config에 명시된 인덱스 컬럼 사용
                df_to_save = df_to_save.set_index(Config.INDEX_COLUMNS)
            else:
                # 단일 인덱스인 경우에도 object 타입에서 Timestamp를 문자열로 변환
                if df_to_save.index.dtype == 'O':
                    df_to_save.index = df_to_save.index.map(
                        lambda x: str(x) if isinstance(x, pd.Timestamp) else x
                    )
            
            df_to_save.to_parquet(self.get_checkpoint_path(column))
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")


    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                checkpoint_df = pd.read_parquet(path)
                
                # 원본 인덱스 형식으로 변환 시도
                if Config.INDEX_COLUMNS and len(Config.INDEX_COLUMNS) > 0:
                    # 인덱스가 이미 있으면 리셋
                    checkpoint_df = checkpoint_df.reset_index()
                    
                    # 날짜 컬럼이 있으면 datetime으로 변환
                    for col in Config.INDEX_COLUMNS:
                        if '날짜' in col and col in checkpoint_df.columns:
                            try:
                                checkpoint_df[col] = pd.to_datetime(checkpoint_df[col])
                            except:
                                pass  # 변환 실패 시 무시
                    
                    # 인덱스 다시 설정
                    checkpoint_df = checkpoint_df.set_index(Config.INDEX_COLUMNS)
                
                return checkpoint_df
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None


#############################################
# 메디컬 텍스트 분류기 클래스 (GPT API 사용)
#############################################

from typing import List, Dict, Optional, Any

class MedicalTextClassifier:
    """의학 텍스트 분류기 클래스"""
    
    def __init__(self, api_key: str, config=None):
        """초기화"""
        self.config = config if config is not None else Config
        
        # API 키 설정 방식 수정
        # 방법 1: 클라이언트 초기화 시 직접 API 키 설정
        self.client = openai.OpenAI(api_key=api_key)  
        
        # 방법 2: 환경 변수로 API 키 설정 (추가적인 방법)
        import os
        os.environ["OPENAI_API_KEY"] = api_key
        self.config = config if config is not None else Config
        self.client = openai.OpenAI(api_key=api_key)  # 클라이언트 객체 생성 방식 변경
        self.semaphore = asyncio.Semaphore(self.config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager(self.config.CHECKPOINT_DIR)
        # Rate Limiter 객체 추가
        self.rate_limiter = RateLimiter(
            rpm_limit=self.config.RPM_LIMIT,
            tpm_limit=self.config.TPM_LIMIT
        )
        # 분류기 메소드 맵핑
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            # 'PI': self._classify_present_illness,
        }
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 컬럼 처리"""
        original_shape = df.shape
        # 원본 인덱스를 보존한 채 복사
        result_df = df.copy()  
        processed_cols = 0
        
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                # 인덱스를 보존한 채 데이터프레임 전달
                column_results = await self._process_column_with_checkpoint(result_df, column)
                
                if column_results is not None and not column_results.empty:
                    # column_results가 원본 인덱스 순서에 맞게 정렬되도록 보장
                    for derived_col in column_results.columns:
                        result_df[derived_col] = column_results[derived_col]
                    
                    derived_cols = [col for col in column_results.columns 
                                    if col.startswith(f"{column}_")]
                    logger.info(f"Column {column} generated {len(derived_cols)} derived columns: {derived_cols}")
                    
                    for derived_col in derived_cols:
                        non_empty_count = result_df[derived_col].notna().sum()
                        logger.info(f"Column {derived_col} has {non_empty_count} non-empty values")
                
                processed_cols += 1
        
        logger.info(f"Original DataFrame shape: {original_shape}, Processed columns: {processed_cols}")
        return result_df
    
    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """체크포인트를 사용한 컬럼 처리"""
        try:
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                logger.info(f"Resumed from checkpoint for {column}")
                # 체크포인트 데이터프레임도 원본 인덱스 순서로 정렬
                try:
                    return checkpoint_df.reindex(df.index)
                except Exception as e:
                    logger.warning(f"체크포인트 인덱스 정렬 실패: {str(e)}. 원본 체크포인트 반환.")
                    return checkpoint_df

            mask = df[column].notna() & (df[column].astype(str).str.strip().str.len() > 0)
            
            if not mask.any():
                logger.info(f"No valid text entries found in column {column}")
                return pd.DataFrame()

            filtered_df = df.loc[mask].copy()
            texts_with_idx = [(idx, text) for idx, text in zip(filtered_df.index, filtered_df[column])]
            
            logger.info(f"Processing {len(texts_with_idx)} valid records for column {column}")
            
            results = await self._safe_process_batches(
                texts=[text for _, text in texts_with_idx],
                original_indices=[idx for idx, _ in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            if results:
                try:
                    # DataFrame 생성 및 인덱스 설정
                    results_df = pd.DataFrame(results)
                    
                    # 'index' 컬럼이 있으면 인덱스로 설정
                    if 'index' in results_df.columns:
                        index_col = results_df.pop('index')
                        
                        # 인덱스 객체를 안전하게 처리
                        index_values = []
                        for idx in index_col:
                            if isinstance(idx, tuple) and len(idx) == 2:
                                # 멀티인덱스 처리
                                patient_id, date = idx
                                # 날짜를 문자열로 변환하여 처리
                                if isinstance(date, pd.Timestamp):
                                    date = date.strftime('%Y-%m-%d')
                                index_values.append((patient_id, date))
                            else:
                                index_values.append(idx)
                        
                        # 안전하게 처리된 인덱스 값으로 인덱스 설정
                        results_df.index = index_values
                    
                    # 컬럼 이름 설정
                    results_df.columns = [f"{column}_{col}" for col in results_df.columns]
                    
                    # 원본 데이터프레임의 인덱스와 정렬 시도
                    try:
                        results_df = results_df.reindex(df.index)
                    except Exception as e:
                        logger.warning(f"결과 데이터프레임 인덱스 정렬 실패: {str(e)}. 원본 결과 반환.")
                    
                    # 체크포인트 저장
                    self.checkpoint.save_checkpoint(results_df, column)
                    return results_df
                except Exception as e:
                    logger.error(f"결과 데이터프레임 생성 중 오류: {str(e)}")
                    # 빈 데이터프레임 반환
                    return pd.DataFrame(index=df.index)

            return pd.DataFrame(index=df.index)

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            logger.error(traceback.format_exc())  # 전체 스택 트레이스 출력
            return pd.DataFrame(index=df.index)  # 오류 시 빈 데이터프레임 반환

    async def _safe_process_batches(self, texts: List[str], original_indices: List[Any],
                                classifier, column: str) -> List[Dict]:
        """각 텍스트를 처리하고 인덱스와 함께 결과 반환"""
        results = []
        
        for i, (text, idx) in enumerate(zip(texts, original_indices)):
            try:
                if not text or not str(text).strip():
                    continue
                    
                logger.info(f"처리 중 {i+1}/{len(texts)} (인덱스 {idx})")
                
                # API 호출
                async with self.semaphore:
                    single_result = await classifier([text], self.semaphore)
                    
                    if single_result and len(single_result) > 0:
                        # 인덱스와 함께 결과 저장
                        result_dict = {"index": idx}
                        result_dict.update(single_result[0])
                        results.append(result_dict)
                    
                    # 중간 저장 (10개마다)
                    if (i + 1) % 10 == 0 or i == len(texts) - 1:
                        if results:
                            try:
                                temp_df = pd.DataFrame(results)
                                # 인덱스 컬럼을 실제 인덱스로 설정
                                if 'index' in temp_df.columns:
                                    # 인덱스 컬럼을 문자열화하여 처리
                                    temp_df = temp_df.copy()
                                    temp_df.set_index('index', inplace=True)
                                    
                                # 컬럼 이름 설정
                                new_cols = {}
                                for col in temp_df.columns:
                                    new_cols[col] = f"{column}_{col}"
                                temp_df = temp_df.rename(columns=new_cols)
                                
                                self.checkpoint.save_checkpoint(temp_df, column)
                            except Exception as e:
                                logger.error(f"중간 체크포인트 저장 중 오류: {str(e)}")
                        
            except Exception as e:
                logger.error(f"인덱스 {idx}의 텍스트 처리 실패: {str(e)}")
        
        return results

    @retry(stop=stop_after_attempt(3),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _process_with_retry(self, classifier, batch_texts: List[str], batch_indices: List[Any]) -> List[Dict]:
        """각 텍스트를 개별적으로 처리하여 정확한 매핑 보장"""
        results = []

        for i, text in enumerate(batch_texts):
            async with self.semaphore:
                # 한 번에 하나의 텍스트 처리
                single_result = await classifier([text], self.semaphore)
                if single_result and len(single_result) > 0:
                    # 결과에 인덱스 추가
                    results.append({"index": batch_indices[i], **single_result[0]})
                else:
                    # API가 빈 또는 유효하지 않은 결과를 반환한 경우 처리
                    results.append({"index": batch_indices[i]})
                
                # 각 개별 결과 로깅
                logger.info(f"텍스트 {i} (인덱스 {batch_indices[i]}) 처리 완료: {single_result[0] if single_result and len(single_result) > 0 else '결과 없음'}")
        
        return results

    def _cleanup_checkpoint(self, column: str) -> None:
        """성공적인 처리 후 체크포인트 정리"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    # o1-mini 모델 사용
    @retry(stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 호출 메소드 (직접 API 호출)"""
        try:
            async with semaphore:
                import httpx
                
                # API 요청 준비
                url = "https://api.openai.com/v1/chat/completions"
                headers = {
                    "Authorization": f"Bearer {self.config.API_KEY}",
                    "Content-Type": "application/json"
                }
                data = {
                    "model": "gpt-3.5-turbo",
                    "messages": [
                        {"role": "system", "content": "JSON 형식으로 응답하세요."},
                        {"role": "user", "content": prompt}
                    ],
                    "max_tokens": self.config.MAX_TOKENS,
                    "temperature": self.config.TEMPERATURE
                }
                
                # API 호출
                async with httpx.AsyncClient() as client:
                    response = await client.post(url, headers=headers, json=data, timeout=60.0)
                    response.raise_for_status()  # 오류 발생 시 예외 발생
                    response_data = response.json()
                
                # 응답 처리
                content = response_data["choices"][0]["message"]["content"]
                logger.debug(f"API Response: {content[:200]}...")
                
                # 토큰 사용량 기록
                total_tokens = response_data["usage"]["total_tokens"]
                self.rate_limiter.record_request(total_tokens)
                
                # JSON 파싱
                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except httpx.HTTPStatusError as e:
            logger.error(f"HTTP error: {e.response.status_code} - {e.response.text}")
            raise
        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """향상된 JSON 응답 검증 및 파싱"""
        try:
            # 원시 내용 로깅
            logger.info(f"원시 API 응답: {content[:500]}...")
            
            # 먼저 직접 JSON 파싱 시도
            try:
                parsed = json.loads(content)
                if isinstance(parsed, list):
                    return parsed
            except json.JSONDecodeError:
                pass  # 직접 파싱이 실패하면 정규식 추출로 계속 진행
            
            # 정규식으로 JSON 추출
            json_pattern = r'```json\s*([\s\S]*?)\s*```|(\[[\s\S]*\])'
            matches = re.findall(json_pattern, content)
            
            for match in matches:
                # 각 일치 항목 시도
                for m in match:
                    if not m.strip():
                        continue
                        
                    try:
                        parsed = json.loads(m.strip())
                        if isinstance(parsed, list):
                            logger.info(f"JSON 파싱 성공: {parsed}")
                            return parsed
                    except:
                        continue
            
            # 마지막 수단: JSON 객체나 배열처럼 보이는 것 찾기
            fallback_pattern = r'(\{[\s\S]*?\}|\[[\s\S]*?\])'
            fallback_matches = re.findall(fallback_pattern, content)
            
            for m in fallback_matches:
                try:
                    parsed = json.loads(m.strip())
                    if isinstance(parsed, dict):
                        # 단일 객체를 목록으로 변환
                        return [parsed]
                    elif isinstance(parsed, list):
                        return parsed
                except:
                    continue
                    
            logger.error(f"응답에서 JSON을 파싱하지 못함")
            return []
            
        except Exception as e:
            logger.error(f"JSON 파싱 오류: {str(e)}")
            return []
        
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Chief Complaints 분류"""
        # 단일 통합 프롬프트 사용
        cc_results = await self._make_api_call(CCPrompts.unified_cc_prompt(texts), semaphore)
        return cc_results
    
    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        results = await self._make_api_call(TreatmentPrompts.medication_prompt(texts), semaphore)
        logger.debug(f"약물 분류 결과: {results}")
        return results

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        results = await self._make_api_call(TreatmentPrompts.device_prompt(texts), semaphore)
        logger.debug(f"장치 분류 결과: {results}")
        return results

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        results = await self._make_api_call(TreatmentPrompts.habit_prompt(texts), semaphore)
        logger.debug(f"습관 분류 결과: {results}")
        return results

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        results = await self._make_api_call(TherapyPrompts.hot_pack_prompt(texts), semaphore)
        logger.debug(f"찜질 분류 결과: {results}")
        return results
    
    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지 및 스트레칭 분류"""
        results = await self._make_api_call(TherapyPrompts.massage_prompt(texts), semaphore)
        logger.debug(f"마사지 분류 결과: {results}")
        return results 
        
#############################################
# CCPrompts 클래스 개선 - 통합 프롬프트 추가
#############################################

class CCPrompts:
    @staticmethod
    def unified_cc_prompt(texts):
        """단일 통합 Chief Complaint 분석 프롬프트"""
        prompt = """
        다음 환자의 Chief Complaint(주호소)를 분석하고 구조화된, 통일된 필드 이름의 JSON 응답을 제공하세요.
        
        특히 다음 항목들을 추출하세요:
        - type: 증상 유형 (예: 통증, 부종, 경직 등)
        - location: 증상 위치 (예: 턱관절, 좌측 귀, 안면부 등)
        - severity: 심각도 (1-10 척도, 가능한 경우)
        - duration: 증상 지속 기간 (예: 3일, 2주, 6개월)
        - first_onset: 최초 발병 시점 (예: 2주 전, 6개월 전)
        - frequency: 발생 빈도 (예: 지속적, 간헐적, 하루에 3회)
        - onset_pattern: 발병 패턴 (예: 갑작스러운, 점진적인)
        - progress: 진행 상태 (예: 악화, 호전, 변화 없음)
        
        다음 주호소를 분석하세요: 
        
        """
        
        for text in texts:
            prompt += f"\n주호소: {text}\n"
        
        prompt += """
        각 주호소에 대해 다음 형식으로 JSON 응답을 제공하세요:
        [
            {
                "type": "주호소 유형",
                "location": "증상 위치",
                "severity": "심각도(가능한 경우 숫자)",
                "duration": "지속 기간",
                "first_onset": "최초 발병 시점",
                "frequency": "발생 빈도",
                "onset_pattern": "발병 패턴",
                "progress": "진행 상태"
            }
        ]
        
        확실하지 않은 정보는 빈 문자열로 남겨두세요. 오직 요청한 필드만 사용하세요.
        """
        
        return prompt
        
    # 기존 프롬프트 유지 (하위 호환성)
    @staticmethod
    def cc_analysis_prompt(texts):
        prompt = """
        다음 환자의 Chief Complaint(주호소)를 분석하고, 증상 유형, 위치, 심각도를 JSON 형식으로 반환해주세요. 
        분명하게 언급되지 않은 정보는 빈 문자열로 남겨두세요.
        """
        
        for text in texts:
            prompt += f"\n주호소: {text}\n"
        
        prompt += """
        각 주호소에 대해 다음 형식으로 JSON 응답을 제공하세요:
        [
            {
                "type": "주호소 유형(예: 통증, 부종, 경직 등)",
                "location": "증상 위치(예: 턱관절, 좌측 귀, 안면부 등)",
                "severity": "심각도(1-10 척도, 가능한 경우)"
            }
        ]
        """
        
        return prompt
    
    @staticmethod
    def cc_history_prompt(texts):
        prompt = """
        다음 환자의 Chief Complaint(주호소)에서 증상의 지속 기간, 최초 발생 시점, 발생 빈도를 추출하여 JSON 형식으로 반환해주세요.
        분명하게 언급되지 않은 정보는 빈 문자열로 남겨두세요.
        """
        
        for text in texts:
            prompt += f"\n주호소: {text}\n"
        
        prompt += """
        각 주호소에 대해 다음 형식으로 JSON 응답을 제공하세요:
        [
            {
                "duration": "증상 지속 기간(예: 3일, 2주, 6개월)",
                "first_onset": "최초 발병 시점(예: 2주 전, 6개월 전)",
                "frequency": "발생 빈도(예: 지속적, 간헐적, 하루에 3회)"
            }
        ]
        """
        
        return prompt
    
    @staticmethod
    def cc_severity_prompt(texts):
        prompt = """
        다음 환자의 Chief Complaint(주호소)에서 증상의 발병 패턴과 진행 상태를 추출하여 JSON 형식으로 반환해주세요.
        분명하게 언급되지 않은 정보는 빈 문자열로 남겨두세요.
        """
        
        for text in texts:
            prompt += f"\n주호소: {text}\n"
        
        prompt += """
        각 주호소에 대해 다음 형식으로 JSON 응답을 제공하세요:
        [
            {
                "onset_pattern": "발병 패턴(예: 갑작스러운, 점진적인)",
                "progress": "진행 상태(예: 악화, 호전, 변화 없음)"
            }
        ]
        """
        
        return prompt


#############################################
# 의학 데이터 처리 함수
#############################################

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        processed_df = await classifier.process_all_columns(df)

        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        for col in processed_columns:
            total_entries = df[col].notna().sum()
            derived_cols = [c for c in processed_df.columns if c.startswith(f"{col}_")]
            success_rate = (len(derived_cols) / (len(derived_cols) + 1) * 100) if derived_cols else 0
            logger.info(f"{col} - Fields extracted: {len(derived_cols)}, Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise


#############################################
# 메인 함수
#############################################

import os
import pandas as pd
import asyncio
import logging
from datetime import datetime
import traceback
from tqdm import tqdm
import glob

def main():
    """메인 함수"""
    global logger
    logger = setup_logging()
    total_start_time = datetime.now()

    try:
        # 원본 데이터 불러오기
        # df = pd.read_excel('환자데이터.xlsx')  # 실제 데이터 경로로 변경
        original_df = df  # 여기서 df는 이미 로드된 데이터프레임
        
        # 인덱스 설정
        logger.info("인덱스 설정 확인 중...")
        if Config.INDEX_COLUMNS and all(col in original_df.columns for col in Config.INDEX_COLUMNS):
            logger.info(f"멀티인덱스 설정: {Config.INDEX_COLUMNS}")
            
            # 멀티인덱스 설정 전 데이터 타입 확인 및 변환
            for col in Config.INDEX_COLUMNS:
                if '날짜' in col and pd.api.types.is_object_dtype(original_df[col]):
                    try:
                        logger.info(f"날짜 컬럼 '{col}' 변환 시도")
                        original_df[col] = pd.to_datetime(original_df[col])
                        logger.info(f"날짜 컬럼 '{col}' 변환 성공")
                    except Exception as e:
                        logger.warning(f"날짜 컬럼 '{col}' 변환 실패: {str(e)}")
            
            # 인덱스 설정 전 원본 인덱스 백업
            original_df['_original_index'] = original_df.index
            
            # 멀티인덱스 설정 (환자번호, 날짜)
            try:
                original_df = original_df.set_index(Config.INDEX_COLUMNS)
                logger.info(f"멀티인덱스 설정 성공: {original_df.index.names}")
            except Exception as e:
                logger.error(f"멀티인덱스 설정 실패: {str(e)}")
                # 실패 시 원본 인덱스로 복원
                original_df = original_df.set_index('_original_index')
                original_df.index.name = None
        else:
            logger.warning(f"인덱스 컬럼을 찾을 수 없거나 완전하지 않습니다: {Config.INDEX_COLUMNS}")
            # 기본 RangeIndex 유지
        
        # 인덱스 확인
        logger.info(f"데이터프레임 인덱스 타입: {type(original_df.index)}")
        logger.info(f"데이터프레임 인덱스 샘플: {original_df.index[:5].tolist() if len(original_df) > 0 else 'No data'}")
        
        # 처리할 총 행 수
        total_rows = len(original_df)
        
        # 청크 크기 설정
        chunk_size = 2000
        
        # 결과 파일을 저장할 디렉토리
        output_dir = 'processed_chunks'
        os.makedirs(output_dir, exist_ok=True)
        
        # 최종 결과 디렉토리
        final_dir = 'final_result'
        os.makedirs(final_dir, exist_ok=True)
        
        # 실패한 레코드를 저장할 데이터프레임
        failed_records = pd.DataFrame(columns=['index', 'column', 'text', 'error'])
        
        # 생성된 파일 경로 목록
        processed_files = []
        
        # API 키 설정
        api_key = Config.API_KEY
        
        # 이벤트 루프 가져오기
        loop = asyncio.get_event_loop()
        
        # 청크 단위로 처리
        for start_idx in tqdm(range(0, total_rows, chunk_size), desc="데이터 청크 처리"):
            end_idx = min(start_idx + chunk_size, total_rows)
            
            # 현재 청크 추출
            chunk_df = original_df.iloc[start_idx:end_idx].copy()
            
            # 청크 번호 계산
            chunk_num = start_idx // chunk_size + 1
            
            try:
                logger.info(f"청크 {chunk_num} 처리 시작 (행 {start_idx+1}-{end_idx})")
                
                # 현재 시간 기록
                chunk_start_time = datetime.now()
                
                # 메디컬 텍스트 분류기 인스턴스 생성 (각 청크마다 새로운 인스턴스)
                classifier = MedicalTextClassifier(api_key)
                
                # 청크 처리
                processed_chunk = loop.run_until_complete(classifier.process_all_columns(chunk_df))
                
                # 실패한 레코드 확인
                for column in classifier.classifiers.keys():
                    if column in chunk_df.columns:
                        derived_cols = [c for c in processed_chunk.columns if c.startswith(f"{column}_")]
                        
                        if derived_cols:  # 파생 필드가 존재하는 경우
                            # 유효한 텍스트가 있지만 처리 결과가 없는 행 찾기
                            mask = chunk_df[column].notna() & chunk_df[column].astype(str).str.strip().astype(bool)
                            has_data_mask = mask.copy()
                            
                            for field in derived_cols:
                                has_data_mask = has_data_mask & processed_chunk[field].isna()
                            
                            failed_indices = processed_chunk[has_data_mask].index.tolist()
                            
                            for idx in failed_indices:
                                failed_records = pd.concat([failed_records, pd.DataFrame([{
                                    'index': idx,
                                    'column': column,
                                    'text': chunk_df.loc[idx, column],
                                    'error': '처리 결과 없음'
                                }])], ignore_index=True)
                                logger.warning(f"레코드 처리 실패: 인덱스 {idx}, 컬럼 {column}, 텍스트: {chunk_df.loc[idx, column][:100]}...")
                
                # 처리 시간 계산
                chunk_end_time = datetime.now()
                chunk_duration = (chunk_end_time - chunk_start_time).total_seconds()
                logger.info(f"청크 {chunk_num} 처리 완료: {chunk_duration:.2f}초 소요 ({(end_idx-start_idx)/chunk_duration:.2f}행/초)")
                
                # parquet 파일로 저장 (인덱스 유지)
                output_file = os.path.join(output_dir, f'chunk_{chunk_num:04d}.parquet')
                processed_chunk.to_parquet(output_file, index=True)
                processed_files.append(output_file)
                logger.info(f"청크 {chunk_num} 결과 저장 완료: {output_file}")
                
            except Exception as e:
                logger.error(f"청크 {chunk_num} 처리 중 오류 발생: {str(e)}")
                logger.error(traceback.format_exc())
                
                # 오류가 발생한 청크의 모든 행을 실패로 기록
                for idx, row in chunk_df.iterrows():
                    for column in row.index:
                        if column in classifier.classifiers and pd.notna(row[column]) and str(row[column]).strip():
                            failed_records = pd.concat([failed_records, pd.DataFrame([{
                                'index': idx,
                                'column': column,
                                'text': row[column],
                                'error': str(e)
                            }])], ignore_index=True)
        
        # 실패한 레코드 저장
        if not failed_records.empty:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            failed_file = os.path.join(output_dir, f'failed_records_{timestamp}.csv')
            failed_records.to_csv(failed_file, index=False, encoding='utf-8-sig')
            logger.info(f"실패한 레코드 {len(failed_records)}개를 {failed_file}에 저장했습니다.")
        
        # 모든 청크 파일 합치기
        logger.info("모든 청크 파일 합치기 시작...")
        
        # 병합 과정에서 인덱스 변환을 위한 함수
        def convert_index_for_merge(df):
            """병합 전 인덱스를 문자열로 변환하여 일관성 있는 인덱스 보장"""
            df_copy = df.copy()
            # 인덱스를 리셋하고 원본 인덱스 열로 변환
            df_copy = df_copy.reset_index()
            
            # 날짜 컬럼을 문자열로 변환
            for col in df_copy.columns:
                if pd.api.types.is_datetime64_any_dtype(df_copy[col]):
                    df_copy[col] = df_copy[col].astype(str)
            
            # 원본 인덱스 컬럼으로 다시 인덱스 설정
            if all(col in df_copy.columns for col in Config.INDEX_COLUMNS):
                df_copy = df_copy.set_index(Config.INDEX_COLUMNS)
            
            return df_copy

        # 각 파일을 순차적으로 읽어서 합치기 (메모리 효율성 고려)
        final_df = None
        for file in tqdm(processed_files, desc="파일 병합"):
            try:
                # 인덱스를 유지하면서 파일 읽기
                chunk = pd.read_parquet(file)
                
                # 읽은 청크의 인덱스 유형 확인
                logger.info(f"청크 인덱스 타입: {type(chunk.index)}")
                logger.info(f"청크 컬럼: {chunk.columns.tolist()}")
                
                # 인덱스 전처리
                chunk = convert_index_for_merge(chunk)
                
                if final_df is None:
                    final_df = chunk
                else:
                    # 병합 전 인덱스 변환 적용
                    final_df = convert_index_for_merge(final_df)
                    # 인덱스를 기준으로 병합 (동일한 인덱스는 누락되지 않음)
                    final_df = pd.concat([final_df, chunk], axis=0)
            except Exception as e:
                logger.error(f"파일 {file} 처리 실패: {str(e)}")
                logger.error(traceback.format_exc())
        
        if final_df is None:
            logger.error("병합할 유효한 데이터가 없습니다.")
            return
        
        # 중복 행 확인 및 제거
        duplicate_rows = final_df.index.duplicated()
        if duplicate_rows.any():
            logger.warning(f"중복된 인덱스가 발견되었습니다: {duplicate_rows.sum()}개")
            # 첫 번째 등장한 인덱스만 유지
            final_df = final_df[~duplicate_rows]
        
        # 최종 결과 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_output = os.path.join(final_dir, f'processed_medical_data_{timestamp}.parquet')
        
        try:
            # 저장 전 인덱스 문자열 변환 (안전한 저장을 위해)
            save_df = final_df.copy()
            save_df = save_df.reset_index()
            
            for col in save_df.columns:
                if pd.api.types.is_datetime64_any_dtype(save_df[col]):
                    save_df[col] = save_df[col].astype(str)
            
            save_df.to_parquet(final_output)
            logger.info(f"최종 결과를 {final_output}에 저장했습니다. (총 {len(save_df)}행)")
            
            # CSV 백업 저장
            csv_output = os.path.join(final_dir, f'processed_medical_data_{timestamp}.csv')
            save_df.to_csv(csv_output, index=False, encoding='utf-8-sig')
            logger.info(f"최종 결과를 CSV 백업 {csv_output}에 저장했습니다.")
        except Exception as e:
            logger.error(f"최종 결과 저장 실패: {str(e)}")
            logger.error(traceback.format_exc())
        
        # 처리 통계 출력
        logger.info("\n=== 처리 결과 요약 ===")
        logger.info(f"총 처리 레코드: {total_rows}")
        logger.info(f"실패한 레코드: {len(failed_records)}")
        success_rate = ((total_rows - len(failed_records)) / total_rows * 100) if total_rows > 0 else 0
        logger.info(f"처리 성공률: {success_rate:.2f}%")
        
        # 파생 컬럼에 대한 통계
        category_columns = list(classifier.classifiers.keys())
        derived_columns = []
        
        for col in category_columns:
            col_derived = [c for c in final_df.columns if c.startswith(f"{col}_")]
            derived_columns.extend(col_derived)
            if col_derived:
                logger.info(f"{col} 파생 컬럼: {len(col_derived)}개")
                for derived in col_derived:
                    valid_values = final_df[derived].notna().sum()
                    logger.info(f"  - {derived}: {valid_values}개 유효값 ({valid_values/len(final_df)*100:.2f}%)")
        
        # 컬럼 구조 확인
        logger.info("\n=== 최종 데이터프레임 구조 ===")
        logger.info(f"총 컬럼 수: {len(final_df.columns)}")
        logger.info(f"원본 컬럼: {[c for c in final_df.columns if c not in derived_columns]}")
        logger.info(f"파생 컬럼 수: {len(derived_columns)}")
        
    except Exception as e:
        logger.error(f"프로그램 실행 중 오류 발생: {str(e)}")
        logger.error(traceback.format_exc())
    finally:
        # 전체 실행 시간 계산
        total_end_time = datetime.now()
        total_duration = (total_end_time - total_start_time).total_seconds()
        hours, remainder = divmod(total_duration, 3600)
        minutes, seconds = divmod(remainder, 60)
        
        logger.info(f"총 실행 시간: {int(hours)}시간 {int(minutes)}분 {seconds:.2f}초")
        if 'total_rows' in locals() and total_duration > 0:
            records_per_second = total_rows / total_duration
            logger.info(f"평균 처리 속도: {records_per_second:.2f}행/초")
        
        logger.info("프로그램 실행 완료")

if __name__ == "__main__":
    main()

2025-03-23 19:21:26,502 - __main__ - INFO - 인덱스 설정 확인 중...
2025-03-23 19:21:26,503 - __main__ - INFO - 멀티인덱스 설정: ['환자번호', '날짜']
2025-03-23 19:21:26,506 - __main__ - INFO - 멀티인덱스 설정 성공: ['환자번호', '날짜']
2025-03-23 19:21:26,507 - __main__ - INFO - 데이터프레임 인덱스 타입: <class 'pandas.core.indexes.multi.MultiIndex'>
2025-03-23 19:21:26,507 - __main__ - INFO - 데이터프레임 인덱스 샘플: [('2312-212', Timestamp('2024-02-02 00:00:00')), ('2112-45', Timestamp('2023-04-25 00:00:00')), ('2309-118', Timestamp('2023-10-12 00:00:00')), ('2304-331', Timestamp('2023-06-27 00:00:00')), ('2207-88', Timestamp('2023-02-18 00:00:00'))]
데이터 청크 처리:   0%|          | 0/1 [00:00<?, ?it/s]2025-03-23 19:21:26,509 - __main__ - INFO - 청크 1 처리 시작 (행 1-100)
2025-03-23 19:21:26,524 - __main__ - INFO - Processing column: CC
2025-03-23 19:21:26,525 - __main__ - INFO - Processing 100 valid records for column CC
2025-03-23 19:21:26,525 - __main__ - INFO - 처리 중 1/100 (인덱스 ('2312-212', Timestamp('2024-02-02 00:00:00')))
2025-03-23 19:21:28,11

In [45]:
pd.read_parquet('/Users/nam-yeong/git/prj_centum/gpt_word/checkpoints/습관_checkpoint.parquet').info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 100 entries, ('2312-212', '2024-02-02 00:00:00') to ('2301-156', '2023-01-27 00:00:00')
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   습관_habit_type   0 non-null      float64
 1   습관_frequency    0 non-null      float64
 2   습관_awareness    0 non-null      float64
 3   습관_improvement  0 non-null      float64
dtypes: float64(4)
memory usage: 9.1+ KB


In [41]:
tt = pd.read_parquet('./final_result/processed_medical_data_20250323_192903.parquet')
tt


,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,_original_index,CC_type,CC_location,CC_severity,CC_duration,CC_first_onset,CC_frequency,CC_onset_pattern,CC_progress,약_medication_type,약_frequency,약_duration,약_compliance,장치_device_type,장치_usage_pattern,장치_duration,장치_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement,찜질_status,찜질_frequency,찜질_duration,찜질_method,"마사지, 스트레칭_type","마사지, 스트레칭_frequency","마사지, 스트레칭_duration","마사지, 스트레칭_method"
0,2312-212,2024-02-02,"물리치료 , 증상 ck구강내과#2/하나증상: 오른쪽 귀앞이 컨디션 안좋을때 붓는게 ...",약: 중간에 통증이 없어져서 1주치 정도 남았어요,None,습관: 질기거나 딱딱한거 안 먹었어요/ 치아끼리 안닿게 턱에 힘 풀고 지냈어요,찜질: 주 2번 10-15분 온찜질햇어요,마사지: 주 2회,None,17161,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
1,2112-45,2023-04-25,"구강내과#17물리치료 , 장치 ck증상: 저번에 주사맞고 오른쪽 통증은 없어졌어요....",None,"장치: 저번에 안좋아졌다고 하셔서 거의 매일 착용했어요, 장치 불편감X","습관: 딱딱하고 질긴 음식은 안먹어요, 치아끼리 닿지 않도록 해요.",찜질: 일주일에 한번정도 찜질팩으로 해요.,None,None,19110,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
2,2309-118,2023-10-12,턱에서 딱딱소리가 나고 통증 있어요초등6학년부터갑자기 하품할때마다 양쪽에서 딱 소...,None,None,None,None,None,#20번대 유구치 mobility+12345678 12345678Dr.남윤진료측두하...,7929,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
3,2304-331,2023-06-27,"구강내과 #3물리치료 , 장치 ck증상: 소리는 빈도수 줄었는데 아직 나기는 나요....",약: 아침약은 다 먹었어요. 저녁약 6개정도 남았어요/ 속쓰림 불편함 없었어요.,장치: APS 1주일에 3번 착용./,습관: 딱딱하거나 질긴음식 안먹었어요./ 치아끼리 안닿도록 노력했어요. 의식적으로 ...,찜질: 아플때만 잠깐잠깐하고 평소에는 잘 안했어요,None,None,4222,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
4,2207-88,2023-02-18,"구강내과 #9물리치료 , 장치 ck증상: 양쪽 벌릴 떄 소리 동일하고 오래 씹으면 ...",None,"장치: 둘 다 위에만 끼는 장치인데 aps2일->ss1일로 계속 착용, 밴드 사용중.",습관: 이랑 이 안 닿게 해요. 질기고 딱딱한 음식 피해요.,찜질: 매일. 10분. 온찜질팩,None,None,10488,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2301-233,2023-04-17,"구강내과 #4물리치료 , 장치 ck증상: 입 크게 벌릴때 저번이랑 비슷한거 같은데 ...",None,장치: 매일착용./ 밴드는 가끔 안할때가 있어서 20장정도 남았어요.,습관: 딱딱하거나 질긴음식 즐겨먹지는 않아요./ 이랑 이가 안닿도록 노력했어요.,찜질: 바쁘고 찜질팩이 터져서 한번도 못했어요,None,None,364,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
96,2209-90,2022-12-08,"구강내과#4물리치료 , 장치 ck증상 : 소리빈도는 점점줄어드는것같아요 ...",None,장치:매일 착용했어요,"습관: 딱딱하고 질긴거 피하려헀어요, 이랑이 안닿게 하는건 신경 못쓴것 같아요",None,None,* 치아잔금 많음,12918,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
97,2209-294,2022-10-31,턱관절) 한달전부터 오른쪽 통증처음엔 음식 처음 씹을 때만 아파서 일반치과 갔더니 ...,None,None,None,None,None,"* 치아 교모 아주 심함* #41 치은퇴축, 염증 심함22-09-30 / BOTOX...",12625,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None
98,2207-217,2022-08-03,"구강내과#2[도착]전체검진, 물리치료 , 이갈이장치 del증상 : 증상은 비슷해요....",None,None,None,None,None,* 구치부 치아 금있음.,9701,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,NaN,None,None,None,None,None


In [13]:
sens = pd.read_parquet('processed_medical_data_20250310_023712.parquet')
nums = pd.read_parquet('../../data/centum_data_numeric_cleaned.parquet')

In [15]:
len(nums), len(sens)


(28108, 28108)

In [19]:
nums.columns

Index(['환자번호', '날짜', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel', 'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_bef

In [22]:
sens = sens[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method']]

In [26]:
fin_df = pd.merge(nums, sens, on=['환자번호','날짜'], how='left')
cols = [
       '환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',
       'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', '치료계획',
       'End feel',
       'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw',
       'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress',
       'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit',
       'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',
       '장치_device_type', '장치_usage_pattern', '장치_duration', '장치_compliance',
       '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement',
       '찜질_status', '찜질_frequency', '찜질_duration', '찜질_method',
       '마사지, 스트레칭_type', '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration',
       '마사지, 스트레칭_method',
       'CMO_before', 'CMO_after', 'MMO_before', 'MMO_after',
       'deviation_pattern_type', 'deviation_direction', 'deviation_intensity',
       'Cap.pal_Pain_Intensity', 'Cap.pal_Pain_Direction',
       'Cap.pal_Pain_Situation', 'M.pal_Pain_Intensity',
       'M.pal_Pain_Direction', 'M.pal_Pain_Situation', 'Noise_Code',
       'Noise_Direction', 'Noise_Intensity', 'Noise_Situation',
       'Occlusion_lt_number', 'Occlusion_rt_number', 'Occlusion_lt_Intensity',
       'Occlusion_rt_Intensity', 'oj', 'ob', 'Midline_Shift_Jaw',
       'Midline_Shift_Direction_x', 'Midline_Shift_Direction_y',
       'Midline_Shift_Amount', 'CRCO_Direction_x', 'CRCO_Direction_y',
       'CRCO_Amount', 'Tongue_ridging_Intensity', 'Rt_before', 'Rt_after',
       'Lt_before', 'Lt_after', 'Next_Visit_Days'
]
fin_df = fin_df[cols]
fin_df.head()

fin_df.to_parquet('../../data/final_without_pi_centum_data_with_medical_data.parquet')
fin_df.to_csv('../../data/final_without_pi_centum_data_with_medical_data.csv', index=False, encoding='utf-8-sig')